# 🦟 Détection et Comptage d'Insectes par Analyse de Trajectoires

Ce notebook permet de détecter et comptabiliser les insectes dans des séquences vidéo infrarouge
en analysant leurs trajectoires pour les différencier des particules en suspension.

## 📋 Principe de fonctionnement
- Les particules suivent un flux régulier (direction du vent)
- Les insectes ont des trajectoires autonomes et erratiques
- Analyse des caractéristiques de mouvement pour la classification


## 1. Installation des dépendances


In [ ]:
# Installation des packages nécessaires
%pip install opencv-python-headless numpy matplotlib scipy scikit-learn pandas tqdm
%pip install filterpy  # Pour le filtre de Kalman

# Import des bibliothèques
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from sklearn.cluster import DBSCAN
from collections import defaultdict, deque
import math
from tqdm import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

print("✅ Bibliothèques installées et importées avec succès!")


## 2. Upload et configuration de la vidéo


In [ ]:
# Pour Google Colab - Upload de fichier vidéo
from google.colab import files
import os

print("📁 Veuillez uploader votre vidéo infrarouge...")
uploaded = files.upload()

# Récupération du nom du fichier
video_filename = list(uploaded.keys())[0]
print(f"✅ Vidéo '{video_filename}' uploadée avec succès!")

# Configuration des paramètres
CONFIG = {
    'video_path': video_filename,
    'output_dir': 'results/',
    
    # Paramètres de détection
    'min_area': 5,  # Aire minimale d'un objet (pixels²)
    'max_area': 500,  # Aire maximale d'un objet
    'threshold_value': 30,  # Seuil pour la détection de mouvement
    
    # Paramètres de tracking
    'max_distance': 50,  # Distance max entre positions (pixels)
    'min_trajectory_length': 10,  # Longueur min de trajectoire (frames)
    'trajectory_buffer': 30,  # Nombre de frames à garder en mémoire
    
    # Paramètres de classification
    'flow_direction_tolerance': 30,  # Tolérance angulaire (degrés)
    'min_velocity_variance': 0.2,  # Variance minimale de vitesse pour un insecte
    'min_direction_changes': 3,  # Changements de direction min pour un insecte
    
    # Paramètres de visualisation
    'show_tracks': True,
    'save_results': True,
    'process_every_n_frames': 1  # Traiter toutes les N frames
}

# Création du dossier de résultats
os.makedirs(CONFIG['output_dir'], exist_ok=True)


## 3. Classes principales pour le tracking


In [ ]:
class TrajectoryAnalyzer:
    """Analyse les trajectoires pour classifier insectes vs particules"""
    
    def __init__(self, config):
        self.config = config
        self.main_flow_direction = None
        self.main_flow_speed = None
        
    def estimate_main_flow(self, trajectories):
        """Estime la direction et vitesse principale du flux (particules)"""
        if not trajectories:
            return
            
        velocities = []
        directions = []
        
        for traj in trajectories:
            if len(traj['positions']) > 5:
                positions = np.array(traj['positions'])
                # Calcul des vecteurs de déplacement
                displacements = np.diff(positions, axis=0)
                
                for disp in displacements:
                    if np.linalg.norm(disp) > 0:
                        velocities.append(np.linalg.norm(disp))
                        directions.append(np.arctan2(disp[1], disp[0]))
        
        if velocities:
            # Direction principale (mode de la distribution circulaire)
            self.main_flow_direction = stats.circmean(directions)
            self.main_flow_speed = np.median(velocities)
            
    def analyze_trajectory(self, trajectory):
        """Analyse une trajectoire et retourne ses caractéristiques"""
        positions = np.array(trajectory['positions'])
        
        if len(positions) < self.config['min_trajectory_length']:
            return None
            
        features = {
            'id': trajectory['id'],
            'length': len(positions),
            'start_frame': trajectory['start_frame'],
            'end_frame': trajectory['start_frame'] + len(positions)
        }
        
        # Calcul des déplacements et vitesses
        displacements = np.diff(positions, axis=0)
        velocities = np.linalg.norm(displacements, axis=1)
        
        # Vitesse moyenne et variance
        features['mean_velocity'] = np.mean(velocities)
        features['velocity_variance'] = np.var(velocities)
        
        # Directions
        directions = []
        for disp in displacements:
            if np.linalg.norm(disp) > 0:
                directions.append(np.arctan2(disp[1], disp[0]))
        
        if directions:
            # Déviation par rapport au flux principal
            if self.main_flow_direction is not None:
                deviations = [self.angular_difference(d, self.main_flow_direction) 
                             for d in directions]
                features['mean_deviation'] = np.mean(np.abs(deviations))
            else:
                features['mean_deviation'] = 0
            
            # Changements de direction
            direction_changes = 0
            for i in range(1, len(directions)):
                angle_diff = self.angular_difference(directions[i], directions[i-1])
                if abs(angle_diff) > np.radians(30):  # Changement > 30°
                    direction_changes += 1
            features['direction_changes'] = direction_changes
            
            # Tortuosité (sinuosité du chemin)
            total_distance = np.sum(velocities)
            straight_distance = np.linalg.norm(positions[-1] - positions[0])
            features['tortuosity'] = total_distance / (straight_distance + 1e-6)
        else:
            features['mean_deviation'] = 0
            features['direction_changes'] = 0
            features['tortuosity'] = 1
            
        return features
    
    def angular_difference(self, angle1, angle2):
        """Calcule la différence entre deux angles (en radians)"""
        diff = angle1 - angle2
        while diff > np.pi:
            diff -= 2 * np.pi
        while diff < -np.pi:
            diff += 2 * np.pi
        return diff
    
    def classify(self, features):
        """Classifie une trajectoire comme insecte ou particule"""
        if features is None:
            return 'unknown'
            
        # Critères de classification pour un insecte:
        is_insect = False
        
        # 1. Variance de vitesse élevée (mouvement erratique)
        if features['velocity_variance'] > self.config['min_velocity_variance']:
            is_insect = True
            
        # 2. Déviation importante par rapport au flux principal
        if features['mean_deviation'] > np.radians(self.config['flow_direction_tolerance']):
            is_insect = True
            
        # 3. Nombreux changements de direction
        if features['direction_changes'] >= self.config['min_direction_changes']:
            is_insect = True
            
        # 4. Trajectoire tortueuse
        if features['tortuosity'] > 1.5:
            is_insect = True
            
        return 'insect' if is_insect else 'particle'


class ObjectTracker:
    """Système de tracking multi-objets"""
    
    def __init__(self, config):
        self.config = config
        self.tracks = {}
        self.next_id = 0
        self.frame_count = 0
        
    def update(self, detections):
        """Met à jour les tracks avec les nouvelles détections"""
        self.frame_count += 1
        
        # Association détections-tracks existants
        unmatched_detections = []
        matched_tracks = set()
        
        for detection in detections:
            best_track = None
            min_distance = self.config['max_distance']
            
            for track_id, track in self.tracks.items():
                if track_id in matched_tracks:
                    continue
                    
                if track['positions']:
                    last_pos = track['positions'][-1]
                    distance = np.linalg.norm(np.array(detection) - np.array(last_pos))
                    
                    if distance < min_distance:
                        min_distance = distance
                        best_track = track_id
            
            if best_track is not None:
                self.tracks[best_track]['positions'].append(detection)
                matched_tracks.add(best_track)
            else:
                unmatched_detections.append(detection)
        
        # Création de nouveaux tracks
        for detection in unmatched_detections:
            self.tracks[self.next_id] = {
                'id': self.next_id,
                'positions': deque([detection], maxlen=self.config['trajectory_buffer']),
                'start_frame': self.frame_count,
                'active': True
            }
            self.next_id += 1
        
        # Désactivation des tracks perdus
        for track_id in self.tracks:
            if track_id not in matched_tracks:
                self.tracks[track_id]['active'] = False
    
    def get_active_tracks(self):
        """Retourne les tracks actifs"""
        return {tid: track for tid, track in self.tracks.items() if track['active']}
    
    def get_completed_tracks(self):
        """Retourne les tracks complétés (pour analyse)"""
        completed = []
        for track in self.tracks.values():
            if len(track['positions']) >= self.config['min_trajectory_length']:
                completed.append({
                    'id': track['id'],
                    'positions': list(track['positions']),
                    'start_frame': track['start_frame']
                })
        return completed


## 4. Détection des objets mobiles


In [ ]:
class MotionDetector:
    """Détecte les objets en mouvement dans la vidéo"""
    
    def __init__(self, config):
        self.config = config
        self.background_subtractor = cv2.createBackgroundSubtractorMOG2(
            detectShadows=False,
            varThreshold=config['threshold_value']
        )
        
    def detect_objects(self, frame):
        """Détecte les objets dans une frame"""
        # Conversion en niveaux de gris si nécessaire
        if len(frame.shape) == 3:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        else:
            gray = frame
            
        # Soustraction de fond
        fg_mask = self.background_subtractor.apply(gray)
        
        # Nettoyage du masque
        kernel = np.ones((3, 3), np.uint8)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)
        
        # Détection des contours
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        detections = []
        for contour in contours:
            area = cv2.contourArea(contour)
            
            if self.config['min_area'] < area < self.config['max_area']:
                # Calcul du centroïde
                M = cv2.moments(contour)
                if M['m00'] != 0:
                    cx = int(M['m10'] / M['m00'])
                    cy = int(M['m01'] / M['m00'])
                    detections.append((cx, cy))
                    
        return detections, fg_mask


## 5. Pipeline de traitement principal


In [ ]:
def process_video(video_path, config):
    """Traite une vidéo complète et retourne les résultats"""
    
    print(f"🎥 Traitement de la vidéo: {video_path}")
    
    # Ouverture de la vidéo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Erreur: Impossible d'ouvrir la vidéo")
        return None
    
    # Informations sur la vidéo
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"📊 Vidéo: {width}x{height} @ {fps}fps, {total_frames} frames")
    
    # Initialisation des composants
    detector = MotionDetector(config)
    tracker = ObjectTracker(config)
    analyzer = TrajectoryAnalyzer(config)
    
    # Variables pour les statistiques
    frame_count = 0
    all_detections = []
    
    # Barre de progression
    pbar = tqdm(total=total_frames, desc="Traitement des frames")
    
    # Traitement frame par frame
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # Traiter uniquement certaines frames pour accélérer
        if frame_count % config['process_every_n_frames'] != 0:
            pbar.update(1)
            continue
            
        # Détection des objets
        detections, fg_mask = detector.detect_objects(frame)
        all_detections.extend(detections)
        
        # Mise à jour du tracking
        tracker.update(detections)
        
        # Visualisation optionnelle (toutes les 30 frames)
        if config['show_tracks'] and frame_count % 30 == 0:
            viz_frame = frame.copy()
            
            # Dessiner les tracks actifs
            for track in tracker.get_active_tracks().values():
                positions = list(track['positions'])
                if len(positions) > 1:
                    for i in range(1, len(positions)):
                        cv2.line(viz_frame, 
                                tuple(map(int, positions[i-1])),
                                tuple(map(int, positions[i])),
                                (0, 255, 0), 2)
            
            # Affichage
            plt.figure(figsize=(15, 5))
            plt.subplot(131)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f'Frame {frame_count}')
            plt.axis('off')
            
            plt.subplot(132)
            plt.imshow(fg_mask, cmap='gray')
            plt.title('Masque de mouvement')
            plt.axis('off')
            
            plt.subplot(133)
            plt.imshow(cv2.cvtColor(viz_frame, cv2.COLOR_BGR2RGB))
            plt.title(f'Tracking ({len(tracker.get_active_tracks())} objets)')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        pbar.update(1)
    
    pbar.close()
    cap.release()
    
    print(f"✅ Traitement terminé: {frame_count} frames analysées")
    
    # Analyse des trajectoires
    print("\n🔍 Analyse des trajectoires...")
    trajectories = tracker.get_completed_tracks()
    
    # Estimation du flux principal (particules)
    analyzer.estimate_main_flow(trajectories)
    
    # Classification des trajectoires
    results = {
        'insects': [],
        'particles': [],
        'unknowns': []
    }
    
    for traj in trajectories:
        features = analyzer.analyze_trajectory(traj)
        if features:
            classification = analyzer.classify(features)
            features['classification'] = classification
            if classification == 'unknown':
                results['unknowns'].append(features)
            else:
                results[classification + 's'].append(features)
    
    # Statistiques
    print(f"\n📈 Résultats de l'analyse:")
    print(f"  • Insectes détectés: {len(results['insects'])}")
    print(f"  • Particules détectées: {len(results['particles'])}")
    print(f"  • Non classifiés: {len(results['unknowns'])}")
    
    if analyzer.main_flow_direction is not None:
        print(f"\n🌬️ Flux principal détecté:")
        print(f"  • Direction: {np.degrees(analyzer.main_flow_direction):.1f}°")
        print(f"  • Vitesse médiane: {analyzer.main_flow_speed:.2f} pixels/frame")
    
    return {
        'video_info': {
            'path': video_path,
            'fps': fps,
            'frames': total_frames,
            'width': width,
            'height': height
        },
        'trajectories': trajectories,
        'results': results,
        'main_flow': {
            'direction': analyzer.main_flow_direction,
            'speed': analyzer.main_flow_speed
        }
    }


## 6. Lancement du traitement


In [ ]:
# Traitement de la vidéo
results = process_video(CONFIG['video_path'], CONFIG)


## 7. Visualisation et analyse des résultats


In [ ]:
def visualize_results(results):
    """Visualise les résultats de l'analyse"""
    
    if not results:
        print("❌ Pas de résultats à visualiser")
        return
    
    # Création d'un DataFrame pour l'analyse
    all_features = []
    for category in ['insects', 'particles']:
        for item in results['results'][category]:
            item['category'] = category
            all_features.append(item)
    
    if not all_features:
        print("⚠️ Aucune trajectoire analysée")
        return
        
    df = pd.DataFrame(all_features)
    
    # Graphiques d'analyse
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # 1. Distribution des vitesses
    if 'mean_velocity' in df.columns:
        ax = axes[0, 0]
        for cat in df['category'].unique():
            data = df[df['category'] == cat]['mean_velocity']
            ax.hist(data, alpha=0.5, label=cat, bins=20)
        ax.set_xlabel('Vitesse moyenne (pixels/frame)')
        ax.set_ylabel('Nombre')
        ax.set_title('Distribution des vitesses')
        ax.legend()
    
    # 2. Variance des vitesses
    if 'velocity_variance' in df.columns:
        ax = axes[0, 1]
        for cat in df['category'].unique():
            data = df[df['category'] == cat]['velocity_variance']
            ax.hist(data, alpha=0.5, label=cat, bins=20)
        ax.set_xlabel('Variance de vitesse')
        ax.set_ylabel('Nombre')
        ax.set_title('Variance des vitesses (erraticité)')
        ax.legend()
    
    # 3. Déviation par rapport au flux
    if 'mean_deviation' in df.columns:
        ax = axes[0, 2]
        for cat in df['category'].unique():
            data = np.degrees(df[df['category'] == cat]['mean_deviation'])
            ax.hist(data, alpha=0.5, label=cat, bins=20)
        ax.set_xlabel('Déviation moyenne (degrés)')
        ax.set_ylabel('Nombre')
        ax.set_title('Déviation par rapport au flux principal')
        ax.legend()
    
    # 4. Changements de direction
    if 'direction_changes' in df.columns:
        ax = axes[1, 0]
        for cat in df['category'].unique():
            data = df[df['category'] == cat]['direction_changes']
            ax.hist(data, alpha=0.5, label=cat, bins=15)
        ax.set_xlabel('Nombre de changements')
        ax.set_ylabel('Fréquence')
        ax.set_title('Changements de direction')
        ax.legend()
    
    # 5. Tortuosité
    if 'tortuosity' in df.columns:
        ax = axes[1, 1]
        for cat in df['category'].unique():
            data = df[df['category'] == cat]['tortuosity']
            ax.hist(data, alpha=0.5, label=cat, bins=20)
        ax.set_xlabel('Tortuosité')
        ax.set_ylabel('Nombre')
        ax.set_title('Tortuosité des trajectoires')
        ax.legend()
    
    # 6. Timeline des détections
    ax = axes[1, 2]
    insects_timeline = [item['start_frame'] for item in results['results']['insects']]
    if insects_timeline:
        ax.hist(insects_timeline, bins=50, color='red', alpha=0.7)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Nombre d\'insectes')
    ax.set_title('Timeline des détections d\'insectes')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/analysis_results.png", dpi=150)
    plt.show()
    
    # Statistiques détaillées
    print("\n📊 Statistiques détaillées:")
    print(df.groupby('category').describe())
    
    return df

# Visualisation
df_results = visualize_results(results)


## 8. Export des résultats


In [ ]:
def export_results(results, config):
    """Exporte les résultats dans différents formats"""
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Export JSON complet
    json_path = f"{config['output_dir']}/results_{timestamp}.json"
    
    # Conversion pour JSON (numpy arrays -> lists)
    json_results = {
        'video_info': results['video_info'],
        'summary': {
            'total_insects': len(results['results']['insects']),
            'total_particles': len(results['results']['particles']),
            'main_flow_direction': float(results['main_flow']['direction']) if results['main_flow']['direction'] else None,
            'main_flow_speed': float(results['main_flow']['speed']) if results['main_flow']['speed'] else None
        },
        'insects': results['results']['insects']
    }
    
    with open(json_path, 'w') as f:
        json.dump(json_results, f, indent=2, default=str)
    print(f"✅ Résultats JSON exportés: {json_path}")
    
    # 2. Export CSV des insectes détectés
    if results['results']['insects']:
        csv_path = f"{config['output_dir']}/insects_{timestamp}.csv"
        df_insects = pd.DataFrame(results['results']['insects'])
        df_insects.to_csv(csv_path, index=False)
        print(f"✅ Liste des insectes exportée: {csv_path}")
    
    # 3. Rapport de synthèse
    report_path = f"{config['output_dir']}/report_{timestamp}.txt"
    with open(report_path, 'w') as f:
        f.write("RAPPORT D'ANALYSE - DÉTECTION D'INSECTES\\n")
        f.write("=" * 50 + "\\n\\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
        f.write(f"Vidéo: {results['video_info']['path']}\\n")
        f.write(f"Durée: {results['video_info']['frames']/results['video_info']['fps']:.1f} secondes\\n\\n")
        
        f.write("RÉSULTATS:\\n")
        f.write("-" * 30 + "\\n")
        f.write(f"Insectes détectés: {len(results['results']['insects'])}\\n")
        f.write(f"Particules détectées: {len(results['results']['particles'])}\\n")
        
        if results['main_flow']['direction']:
            f.write(f"\\nFlux principal:\\n")
            f.write(f"  Direction: {np.degrees(results['main_flow']['direction']):.1f}°\\n")
            f.write(f"  Vitesse: {results['main_flow']['speed']:.2f} px/frame\\n")
        
        # Détail des insectes
        if results['results']['insects']:
            f.write(f"\\nDÉTAIL DES INSECTES:\\n")
            f.write("-" * 30 + "\\n")
            for i, insect in enumerate(results['results']['insects'], 1):
                f.write(f"\\nInsecte #{i}:\\n")
                f.write(f"  Frames: {insect['start_frame']} - {insect['end_frame']}\\n")
                f.write(f"  Durée: {(insect['end_frame']-insect['start_frame'])/results['video_info']['fps']:.1f}s\\n")
                f.write(f"  Vitesse moy: {insect['mean_velocity']:.2f} px/frame\\n")
                f.write(f"  Changements direction: {insect['direction_changes']}\\n")
    
    print(f"✅ Rapport de synthèse généré: {report_path}")
    
    return {
        'json_path': json_path,
        'csv_path': csv_path if results['results']['insects'] else None,
        'report_path': report_path
    }

# Export des résultats
export_paths = export_results(results, CONFIG)


## 9. Traitement en batch (plusieurs vidéos)


In [ ]:
def process_multiple_videos(video_list, config):
    """Traite plusieurs vidéos et compile les résultats"""
    
    all_results = []
    total_insects = 0
    
    for video_path in video_list:
        print(f"\n{'='*60}")
        print(f"Traitement de: {video_path}")
        print(f"{'='*60}")
        
        try:
            results = process_video(video_path, config)
            if results:
                all_results.append(results)
                total_insects += len(results['results']['insects'])
                
        except Exception as e:
            print(f"❌ Erreur lors du traitement: {e}")
            continue
    
    # Synthèse globale
    print(f"\n{'='*60}")
    print(f"SYNTHÈSE GLOBALE")
    print(f"{'='*60}")
    print(f"Vidéos traitées: {len(all_results)}/{len(video_list)}")
    print(f"Total insectes détectés: {total_insects}")
    
    # Graphique de synthèse
    if all_results:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Nombre d'insectes par vidéo
        video_names = [r['video_info']['path'].split('/')[-1] for r in all_results]
        insect_counts = [len(r['results']['insects']) for r in all_results]
        
        ax1.bar(range(len(video_names)), insect_counts)
        ax1.set_xticks(range(len(video_names)))
        ax1.set_xticklabels(video_names, rotation=45, ha='right')
        ax1.set_ylabel('Nombre d\'insectes')
        ax1.set_title('Insectes détectés par vidéo')
        
        # Distribution temporelle (si plusieurs vidéos d'une nuit)
        all_frames = []
        for r in all_results:
            for insect in r['results']['insects']:
                all_frames.append(insect['start_frame'])
        
        if all_frames:
            ax2.hist(all_frames, bins=30, color='red', alpha=0.7)
            ax2.set_xlabel('Frame')
            ax2.set_ylabel('Fréquence')
            ax2.set_title('Distribution temporelle des insectes')
        
        plt.tight_layout()
        plt.savefig(f"{config['output_dir']}/batch_summary.png", dpi=150)
        plt.show()
    
    return all_results

# Pour traiter plusieurs vidéos (décommentez et modifiez):
# video_list = ['video1.mp4', 'video2.mp4', 'video3.mp4']
# batch_results = process_multiple_videos(video_list, CONFIG)


## 10. Optimisations et améliorations futures

### 🚀 Pistes d'amélioration:

1. **Utilisation de l'Optical Flow dense** pour mieux caractériser le flux principal
2. **Deep Learning (YOLO)** pour une détection plus robuste des insectes
3. **Filtre de Kalman** pour un tracking plus précis
4. **Clustering spatial** pour gérer les essaims
5. **Analyse spectrale** des trajectoires
6. **Machine Learning** pour la classification (Random Forest, SVM)

### 📈 Métriques supplémentaires possibles:

- Temps de résidence dans le champ de vision
- Patterns de vol spécifiques aux espèces
- Interactions entre individus
- Zones d'attraction (hotspots)
- Analyse circadienne (variation horaire)


In [ ]:
print("\n✅ Analyse terminée avec succès!")
print(f"📁 Les résultats sont disponibles dans: {CONFIG['output_dir']}")
print("\n🔍 Pour analyser d'autres vidéos, modifiez CONFIG['video_path'] et relancez le notebook.")


## 11. Améliorations basées sur les observations terrain

### 📝 Observations clés identifiées :

1. **Bruit de fond** : "étoiles filantes" = poussières proches capteur
   - Direction similaire, floues, peu brillantes, trajectoire rectiligne

2. **Caractéristiques insectes** :
   - Signal lumineux plus ponctuel
   - Trajectoires aléatoires (aller-retour possibles)
   - Clignotements (battements d'ailes) → fréquence mesurable
   - Vol stationnaire possible
   - Taille très variable (très petits à gros objets)

3. **Conditions environnementales** :
   - Pluie fine : diagonales depuis même zone (facile à discriminer)
   - Lune peut perturber l'image
   - Plus d'activité en début de soirée


In [ ]:
class EnhancedTrajectoryAnalyzer(TrajectoryAnalyzer):
    """Version améliorée basée sur les observations terrain"""
    
    def __init__(self, config):
        super().__init__(config)
        self.rain_detected = False
        self.rain_direction = None
        
    def detect_rain_patterns(self, trajectories):
        """Détecte les motifs de pluie (trajectoires parallèles depuis même zone)"""
        if len(trajectories) < 5:
            return
            
        # Analyser les directions de départ
        start_directions = []
        start_positions = []
        
        for traj in trajectories:
            if len(traj['positions']) > 3:
                positions = np.array(traj['positions'])
                start_pos = positions[0]
                direction_vector = positions[3] - positions[0]
                
                if np.linalg.norm(direction_vector) > 0:
                    direction = np.arctan2(direction_vector[1], direction_vector[0])
                    start_directions.append(direction)
                    start_positions.append(start_pos)
        
        if len(start_directions) > 5:
            # Vérifier si les directions sont similaires (pluie)
            direction_variance = np.var(start_directions)
            if direction_variance < 0.1:  # Directions très similaires
                self.rain_detected = True
                self.rain_direction = np.mean(start_directions)
                print("🌧️ Motif de pluie détecté!")
    
    def analyze_trajectory_enhanced(self, trajectory, frame_intensities=None):
        """Analyse enrichie d'une trajectoire"""
        # Analyse de base
        features = self.analyze_trajectory(trajectory)
        if features is None:
            return None
        
        positions = np.array(trajectory['positions'])
        
        # 1. Analyse de la netteté (approximation par variation de position)
        displacements = np.diff(positions, axis=0)
        displacement_norms = np.linalg.norm(displacements, axis=1)
        features['smoothness'] = np.var(displacement_norms)  # Plus c'est lisse, plus c'est flou
        
        # 2. Détection de vol stationnaire
        if len(positions) > 10:
            # Zone d'occupation (convex hull area)
            from scipy.spatial import ConvexHull
            try:
                if len(np.unique(positions, axis=0)) > 2:  # Au moins 3 points uniques
                    hull = ConvexHull(positions)
                    features['occupied_area'] = hull.volume  # Area en 2D
                else:
                    features['occupied_area'] = 0
            except:
                features['occupied_area'] = 0
                
            # Vol stationnaire si petite zone + longue durée
            trajectory_length = np.sum(displacement_norms)
            straight_distance = np.linalg.norm(positions[-1] - positions[0])
            features['stationary_ratio'] = straight_distance / (trajectory_length + 1e-6)
        else:
            features['occupied_area'] = 0
            features['stationary_ratio'] = 1
            
        # 3. Analyse des clignotements (si intensités disponibles)
        if frame_intensities and len(frame_intensities) > 5:
            # FFT pour détecter les fréquences de clignotement
            from scipy.fft import fft, fftfreq
            
            intensities = np.array(frame_intensities)
            # Normaliser
            intensities = (intensities - np.mean(intensities)) / (np.std(intensities) + 1e-6)
            
            # FFT
            fft_values = np.abs(fft(intensities))
            freqs = fftfreq(len(intensities))
            
            # Chercher des pics de fréquence (battements d'ailes)
            # Fréquence typique insectes : 10-1000 Hz (mais échantillonnage vidéo limité)
            peak_freq_idx = np.argmax(fft_values[1:len(fft_values)//2]) + 1
            features['dominant_frequency'] = abs(freqs[peak_freq_idx])
            features['flicker_strength'] = np.max(fft_values[1:len(fft_values)//2])
        else:
            features['dominant_frequency'] = 0
            features['flicker_strength'] = 0
            
        # 4. Détection de passage proche rapide
        if len(displacement_norms) > 0:
            max_speed = np.max(displacement_norms)
            mean_speed = np.mean(displacement_norms)
            features['speed_burst_ratio'] = max_speed / (mean_speed + 1e-6)
        else:
            features['speed_burst_ratio'] = 1
            
        return features
    
    def classify_enhanced(self, features):
        """Classification améliorée"""
        if features is None:
            return 'unknown'
        
        # Critères de classification
        confidence_scores = {
            'insect': 0,
            'dust': 0,
            'rain': 0,
            'artifact': 0
        }
        
        # 1. Détection de pluie
        if self.rain_detected and features['mean_deviation'] < np.radians(15):
            confidence_scores['rain'] += 3
            
        # 2. Caractéristiques des insectes
        # Signal ponctuel et net
        if features['smoothness'] < 5:  # Mouvement net
            confidence_scores['insect'] += 1
            
        # Trajectoire erratique
        if features['velocity_variance'] > 0.5:
            confidence_scores['insect'] += 2
            
        # Changements de direction
        if features['direction_changes'] >= 3:
            confidence_scores['insect'] += 2
            
        # Vol stationnaire
        if features['stationary_ratio'] < 0.3 and features['length'] > 15:
            confidence_scores['insect'] += 2
            
        # Clignotements (battements d'ailes)
        if features['flicker_strength'] > 0.1:
            confidence_scores['insect'] += 2
            
        # Passage rapide proche
        if features['speed_burst_ratio'] > 3:
            confidence_scores['insect'] += 1
            
        # 3. Caractéristiques des poussières
        # Mouvement rectiligne et régulier
        if (features['velocity_variance'] < 0.2 and 
            features['direction_changes'] < 2 and 
            features['tortuosity'] < 1.2):
            confidence_scores['dust'] += 2
            
        # Trajectoire floue/lisse
        if features['smoothness'] > 10:
            confidence_scores['dust'] += 1
            
        # 4. Classification finale
        best_class = max(confidence_scores.keys(), key=lambda x: confidence_scores[x])
        max_score = confidence_scores[best_class]
        
        # Seuil de confiance
        if max_score < 2:
            return 'unknown'
        elif best_class in ['rain', 'dust']:
            return 'particle'
        else:
            return best_class


class EnhancedMotionDetector(MotionDetector):
    """Détecteur amélioré avec analyse d'intensité"""
    
    def detect_objects_enhanced(self, frame):
        """Détection avec informations d'intensité"""
        detections, fg_mask = self.detect_objects(frame)
        
        # Extraction des intensités pour chaque détection
        if len(frame.shape) == 3:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        else:
            gray = frame
            
        intensities = []
        for detection in detections:
            x, y = int(detection[0]), int(detection[1])
            # Zone 5x5 autour du point
            region = gray[max(0, y-2):min(gray.shape[0], y+3), 
                         max(0, x-2):min(gray.shape[1], x+3)]
            if region.size > 0:
                intensities.append(np.mean(region))
            else:
                intensities.append(0)
                
        return detections, fg_mask, intensities


In [ ]:
def process_video_enhanced(video_path, config):
    """Version améliorée du traitement vidéo"""
    
    print(f"🎥 Traitement amélioré de la vidéo: {video_path}")
    
    # Ouverture de la vidéo
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Erreur: Impossible d'ouvrir la vidéo")
        return None
    
    # Informations sur la vidéo
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    print(f"📊 Vidéo: {width}x{height} @ {fps}fps, {total_frames} frames")
    
    # Initialisation des composants améliorés
    detector = EnhancedMotionDetector(config)
    tracker = ObjectTracker(config)
    analyzer = EnhancedTrajectoryAnalyzer(config)
    
    # Stockage des intensités par track
    track_intensities = defaultdict(list)
    
    # Variables pour les statistiques
    frame_count = 0
    
    # Barre de progression
    pbar = tqdm(total=total_frames, desc="Traitement des frames")
    
    # Traitement frame par frame
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        if frame_count % config['process_every_n_frames'] != 0:
            pbar.update(1)
            continue
            
        # Détection des objets avec intensités
        detections, fg_mask, intensities = detector.detect_objects_enhanced(frame)
        
        # Mise à jour du tracking
        tracker.update(detections)
        
        # Stockage des intensités par track actif
        active_tracks = tracker.get_active_tracks()
        for i, (track_id, track) in enumerate(active_tracks.items()):
            if i < len(intensities) and track['positions']:
                track_intensities[track_id].append(intensities[i])
        
        # Visualisation optionnelle (toutes les 30 frames)
        if config['show_tracks'] and frame_count % 30 == 0:
            viz_frame = frame.copy()
            
            # Dessiner les tracks actifs avec couleurs différentes
            colors = [(0, 255, 0), (255, 0, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]
            for i, track in enumerate(active_tracks.values()):
                color = colors[i % len(colors)]
                positions = list(track['positions'])
                if len(positions) > 1:
                    for j in range(1, len(positions)):
                        cv2.line(viz_frame, 
                                tuple(map(int, positions[j-1])),
                                tuple(map(int, positions[j])),
                                color, 2)
                    
                    # Numéro du track
                    cv2.putText(viz_frame, f"T{track['id']}", 
                              tuple(map(int, positions[-1])), 
                              cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            
            # Affichage avec infos supplémentaires
            plt.figure(figsize=(15, 5))
            plt.subplot(131)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title(f'Frame {frame_count} - {len(detections)} détections')
            plt.axis('off')
            
            plt.subplot(132)
            plt.imshow(fg_mask, cmap='gray')
            plt.title('Masque de mouvement')
            plt.axis('off')
            
            plt.subplot(133)
            plt.imshow(cv2.cvtColor(viz_frame, cv2.COLOR_BGR2RGB))
            plt.title(f'Tracking ({len(active_tracks)} tracks actifs)')
            plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
        pbar.update(1)
    
    pbar.close()
    cap.release()
    
    print(f"✅ Traitement terminé: {frame_count} frames analysées")
    
    # Analyse des trajectoires avec détection de pluie
    print("\n🔍 Analyse avancée des trajectoires...")
    trajectories = tracker.get_completed_tracks()
    
    # 1. Détection des motifs de pluie
    analyzer.detect_rain_patterns(trajectories)
    
    # 2. Estimation du flux principal
    analyzer.estimate_main_flow(trajectories)
    
    # 3. Classification enrichie
    results = {
        'insects': [],
        'particles': [],
        'unknowns': [],
        'rain': []
    }
    
    for traj in trajectories:
        # Récupération des intensités pour cette trajectoire
        intensities = track_intensities.get(traj['id'], None)
        
        # Analyse enrichie
        features = analyzer.analyze_trajectory_enhanced(traj, intensities)
        if features:
            classification = analyzer.classify_enhanced(features)
            features['classification'] = classification
            
            # Ajout des métriques spéciales
            if intensities:
                features['mean_intensity'] = np.mean(intensities)
                features['intensity_variance'] = np.var(intensities)
            
            # Classification
            if classification == 'insect':
                results['insects'].append(features)
            elif classification == 'rain':
                results['rain'].append(features)
            elif classification == 'unknown':
                results['unknowns'].append(features)
            else:  # particle, dust, artifact
                results['particles'].append(features)
    
    # Statistiques détaillées
    print(f"\n📈 Résultats de l'analyse avancée:")
    print(f"  • 🦟 Insectes détectés: {len(results['insects'])}")
    print(f"  • 💨 Particules/poussières: {len(results['particles'])}")
    print(f"  • 🌧️ Gouttes de pluie: {len(results['rain'])}")
    print(f"  • ❓ Non classifiés: {len(results['unknowns'])}")
    
    if analyzer.main_flow_direction is not None:
        print(f"\n🌬️ Flux principal détecté:")
        print(f"  • Direction: {np.degrees(analyzer.main_flow_direction):.1f}°")
        print(f"  • Vitesse médiane: {analyzer.main_flow_speed:.2f} pixels/frame")
    
    if analyzer.rain_detected:
        print(f"\n🌧️ Pluie détectée:")
        print(f"  • Direction: {np.degrees(analyzer.rain_direction):.1f}°")
    
    # Analyse des battements d'ailes détectés
    wing_beats = [f['dominant_frequency'] * fps for f in results['insects'] 
                  if f.get('dominant_frequency', 0) > 0.01]
    if wing_beats:
        print(f"\n🦋 Fréquences de battements détectées:")
        print(f"  • Moyenne: {np.mean(wing_beats):.1f} Hz")
        print(f"  • Plage: {np.min(wing_beats):.1f} - {np.max(wing_beats):.1f} Hz")
    
    return {
        'video_info': {
            'path': video_path,
            'fps': fps,
            'frames': total_frames,
            'width': width,
            'height': height
        },
        'trajectories': trajectories,
        'results': results,
        'main_flow': {
            'direction': analyzer.main_flow_direction,
            'speed': analyzer.main_flow_speed
        },
        'environmental': {
            'rain_detected': analyzer.rain_detected,
            'rain_direction': analyzer.rain_direction
        },
        'wing_beat_frequencies': wing_beats if 'wing_beats' in locals() else []
    }


## 12. Test de la version améliorée

### 🚀 Nouvelles fonctionnalités :

- **Détection de pluie** : Trajectoires parallèles depuis même zone
- **Analyse des clignotements** : Fréquences de battements d'ailes
- **Vol stationnaire** : Détection des insectes qui planent
- **Classification multi-critères** : Score de confiance pour chaque catégorie
- **Analyse d'intensité** : Discrimination signal ponctuel vs diffus


In [ ]:
# Test de la version améliorée
print("🔬 Lancement de l'analyse améliorée...")
results_enhanced = process_video_enhanced(CONFIG['video_path'], CONFIG)

# Comparaison avec les résultats de base si disponibles
if 'results' in locals():
    print(f"\n📊 Comparaison des résultats:")
    print(f"Version de base    : {len(results['results']['insects'])} insectes")
    print(f"Version améliorée  : {len(results_enhanced['results']['insects'])} insectes")
    
    print(f"\n🆕 Nouvelles détections:")
    print(f"  • Gouttes de pluie : {len(results_enhanced['results']['rain'])}")
    print(f"  • Battements d'ailes : {len(results_enhanced['wing_beat_frequencies'])} insectes")
    
# Affichage détaillé pour les insectes avec battements d'ailes
wing_beat_insects = [f for f in results_enhanced['results']['insects'] 
                     if f.get('flicker_strength', 0) > 0.1]

if wing_beat_insects:
    print(f"\n🦋 Insectes avec battements d'ailes détectés:")
    for i, insect in enumerate(wing_beat_insects[:5]):  # Top 5
        freq_hz = insect.get('dominant_frequency', 0) * results_enhanced['video_info']['fps']
        print(f"  Insecte #{insect['id']}: {freq_hz:.1f} Hz (frames {insect['start_frame']}-{insect['end_frame']})")

# Insectes en vol stationnaire
stationary_insects = [f for f in results_enhanced['results']['insects'] 
                     if f.get('stationary_ratio', 1) < 0.3]

if stationary_insects:
    print(f"\n🚁 Insectes en vol stationnaire: {len(stationary_insects)}")
    for insect in stationary_insects[:3]:  # Top 3
        print(f"  Insecte #{insect['id']}: ratio {insect['stationary_ratio']:.3f}")

print(f"\n✅ Analyse améliorée terminée!")
print(f"📁 Les résultats sont dans la variable 'results_enhanced'")


## 🎯 Guide d'utilisation selon les observations terrain

### 📋 Checklist avant analyse :

1. **Conditions météo** :
   - ☂️ Si pluie visible → L'algorithme la détectera automatiquement
   - 🌙 Si lune présente → Peut saturer l'image, orienter différemment si possible

2. **Paramètres recommandés selon vidéo** :

```python
# Pour vidéos avec beaucoup de poussières (ex: début soirée)
CONFIG_DUSTY = CONFIG.copy()
CONFIG_DUSTY.update({
    'min_area': 3,  # Capturer les très petits insectes
    'threshold_value': 25,  # Plus sensible
    'min_trajectory_length': 8,  # Trajectoires plus courtes OK
})

# Pour vidéos avec lune (image saturée)
CONFIG_MOON = CONFIG.copy()
CONFIG_MOON.update({
    'threshold_value': 40,  # Moins sensible aux variations
    'min_area': 10,  # Objets plus gros seulement
})
```

### 🔍 Interprétation des résultats :

- **Fréquences battements** 10-50 Hz → Probablement moucherons/petits diptères
- **Fréquences battements** 50-200 Hz → Probablement mouches moyennes  
- **Vol stationnaire** + clignotements → Insectes à ailes transparentes
- **Trajectoire chaotique** → Comportement de chasse ou d'évitement
- **Passage rapide flou** → Insecte très proche du capteur

### 📹 Vidéos mentionnées à analyser :
- `DSCF0817` (8-10s) : Très petits insectes
- `DSCF0820` (18-20s) : Trajectoire chaotique
- `DSCF0814` : Clignotements (battements d'ailes)
- `DSCF0816` : Vol stationnaire
- `DSCF0905` : Pluie fine
- `DSCF0911` : Passage proche rapide
